In [ ]:
import os
import dotenv
import sqlalchemy
import pandas as pd

dotenv.load_dotenv(dotenv_path="cred.env", override=True)

username = os.getenv("username") 
password = os.getenv("password") 
host = os.getenv("host") 
dbname = os.getenv("dbname") 

connection_string = "mysql+pymysql://"+username+":"+password+"@"+host+"/"+dbname 

db_engine = sqlalchemy.create_engine(connection_string)

query = "SELECT * FROM dimproduct"
dimproduct = pd.read_sql(query, db_engine) 
dimproduct.sample(5)

In [ ]:
#Arrotondo la colonna DealerPrice a 2 decimali e poi al valore intero più vicino. 
dimproduct["DealerPrice"] = dimproduct["DealerPrice"].round(2).round()
dimproduct["DealerPrice"]


In [ ]:
#Faccio in modo che i valori siano compresi tra un minimo di 0 e un massimo di 1000
dimproduct["DealerPrice"] = dimproduct["DealerPrice"].clip(lower = 0, upper = 1000)
dimproduct["DealerPrice"]

In [ ]:
import pandas as pd
import numpy as np
years = 5 
guadagni = pd.DataFrame({
    "Mese": list("GFMAMGLASOND"*years), 
    "Anno": np.repeat(list(range(years)), 12), 
    "Valore": np.random.randint(800, 5000, 12*years)
})
guadagni.head(5)

In [ ]:
guadagni["somma_globale"] = guadagni["Valore"].cumsum() #somma cumulativa guadagni

guadagni["somma_per_anno"] = guadagni.groupby("Anno")["Valore"].cumsum() #somma cumnulativa guadagni raggruppato per ogni anno

guadagni.head(15)

In [ ]:
query = "SELECT * FROM dimcustomer"
dimcustomer = pd.read_sql(query, db_engine) 
dimcustomer.head(5)

In [ ]:
#Trasformo i nomi dei clienti in lettere minuscole, e i cognomi in lettere maiuscole
dimcustomer["FirstName"] = dimcustomer["FirstName"].str.lower()
dimcustomer["LastName"] = dimcustomer["LastName"].str.upper()
dimcustomer.head(5)

In [ ]:
#Estraggo il nome utente e dominio
dimcustomer[["user_name", "email_domain"]] = dimcustomer["EmailAddress"].str.split("@", expand = True)

dimcustomer[["EmailAddress", "user_name", "email_domain"]]

In [ ]:
#Estraggo ogni parte del numero (ad es. da "1 (11) 500 555-0162" a ["1", "(11)", "500", "555-0162"])
dimcustomer["Phone"] = dimcustomer["Phone"].str.split(" ")
dimcustomer["Phone"]

In [ ]:
#Estraggo tutti gli indirizzi e-mail che contengono il numero "21" 
email_filtrate = dimcustomer["EmailAddress"].str.contains("21")
dimcustomer[email_filtrate]["EmailAddress"]

In [ ]:
#Estraggo tutti gli indirizzi e-mail che contengono il numero "20" oppure il numero "10" 
email_filtrate_2010 = dimcustomer["EmailAddress"].str.contains("20|10")
dimcustomer[email_filtrate_2010]["EmailAddress"]

In [ ]:
#Calcolo la lunghezza di ogni indirizzo e-mail ed estraggo i cinque più lunghi e i cinque più corti
dimcustomer["lunghezza_email"] = dimcustomer["EmailAddress"].str.len()
print("5 Email più corte: ")
print(dimcustomer.sort_values(by= "lunghezza_email")[["EmailAddress","lunghezza_email"]].head(5))
print("5 Email più lunghe: ")
print(dimcustomer.sort_values(by= "lunghezza_email")[["EmailAddress","lunghezza_email"]].tail(5))

In [ ]:
#Modifico il dominio degli indirizzi e-mail da "adventure-works.com" a "aw-db.com"
dimcustomer["EmailAddress"] = dimcustomer["EmailAddress"].str.replace("adventure-works.com", "aw-db.com")
dimcustomer["EmailAddress"]

In [ ]:
#Estraggo tutti gli indirizzi che contengono la sottostringa "Street"
Indirizzo_filtrato = dimcustomer["AddressLine1"].str.contains("Street")
dimcustomer["AddressLine1"][Indirizzo_filtrato]

In [ ]:
import pandas as pd
facebook_df = pd.read_csv("~/Desktop/Python - Epicode/datasets/beginner_datasets/facebook.csv")
facebook_df

In [ ]:
#Converto la colonna status_published in formato Timestamp
facebook_df["status_published"]= pd.to_datetime(facebook_df["status_published"])
facebook_df["status_published"]

In [ ]:
#Ottengo informazioni specifiche sulle date delle transazioni, come l'anno, il mese, il giorno della settimana, il giorno dell'anno.
facebook_df["year"] = facebook_df["status_published"].dt.year
facebook_df["month"] = facebook_df["status_published"].dt.month
facebook_df["day"] = facebook_df["status_published"].dt.day
facebook_df["dayofweek"] = facebook_df["status_published"].dt.dayofweek
facebook_df["dayofyear"] = facebook_df["status_published"].dt.dayofyear

facebook_df[["status_published", "year", "month", "day", "dayofweek", "dayofyear"]]


In [ ]:
#Estraggo solo i post relativi al 2012 
facebook_df[facebook_df["year"] == 2012].sample(5)

In [ ]:
#Estraggo solo i post relativi a maggio 2018
facebook_df[(facebook_df["year"] == 2018) & (facebook_df["month"] == 5)].sample(5)

In [ ]:
#Confronto il numero di post pubblicati nei weekend rispetto al numero di post pubblicati nel resto della settimana
facebook_df["is_weekend"] = facebook_df["dayofweek"].isin([5, 6])
count = facebook_df["is_weekend"].value_counts()

print("Post nei giorni feriali: ", count[False])
print("Post nei week end ", count[True])

In [ ]:
#Trovo il primo e ultimo post pubblicati in ogni anno
facebook_df.groupby("year")["status_published"].agg(["min", "max"])

In [ ]:
#Quanti tipi di post ci sono? E quanti per ogni tipo?
print("Tipi di post diversi:", facebook_df["status_type"].nunique())
print("Conteggio per tipo:")
print(facebook_df["status_type"].value_counts())

In [ ]:
import pandas as pd
pokemon_df = pd.read_csv("~/Desktop/Python - Epicode/datasets/beginner_datasets/pokemon.csv")
pokemon_df

In [ ]:
#Controllo se ci sono e quanti sono i valori nulli per ogni colonna
pokemon_df.isnull().sum()

In [ ]:
#Ci sono valori nulli? • Se sì, avrebbe senso cercare di riempirli?
print("Si, ci sono valori nulli. Ma non ha senso riempirlo in quanto non tutti i pokemnon hanno un secondo tipo.")

In [ ]:
#Elimino i valori nulli
pokemon_no_null = pokemon_df.dropna()
pokemon_no_null.isnull().sum()

In [ ]:
import pandas as pd
automobile_df = pd.read_csv("~/Desktop/Python - Epicode/datasets/beginner_datasets/automobile.csv")
automobile_df

In [ ]:
# Ci sono valori nulli? Dove? Quanti?
print("Si, ci sono valori nulli nella colonna \"normalized-losses\" e \"num-of-doors\"")
print(automobile_df.isnull().sum())

In [ ]:
#Quali righe hanno un valore nullo nella colonna num-of-doors?
automobile_df[automobile_df["num-of-doors"].isnull()]

In [ ]:
#Esaminando i dati nel dataset, cerchiamo una logica per sostituire i valori nulli nella colonna num-of-doors

print(automobile_df.groupby("body-style")["num-of-doors"].value_counts()) #raggruppo per tipo di macchina e vedo quante porte hanno

print(automobile_df.groupby("make")["num-of-doors"].value_counts()) #raggruppo per marca e vedo quante porte hanno

# Guarda le auto nulle - che body-style hanno
auto_nulle = automobile_df[automobile_df["num-of-doors"].isnull()]
print(auto_nulle[["make", "body-style"]])

#Riempo i valori nulli in num-of-doors con "four" perché la maggioranza di sedan ha 4 porte
automobile_df["num-of-doors"] = automobile_df["num-of-doors"].fillna("four")

In [ ]:
import numpy as np
import pandas as pd 

temp = pd.DataFrame({
    "Giorno": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 
    "Temperature": [18, 19, 18, np.nan, 21, 20, 20, np.nan, 21, 23, np.nan, 23, 24]
})

temp


In [ ]:
# Il sensore a volte non funziona, dunque alcuni dati sono mancanti: quale sarebbe la migliore strategia per gestirli? 
media_temperature = temp["Temperature"].mean() #calcolo la media totale
print(media_temperature)

temp["Temperature"] = temp["Temperature"].fillna(media_temperature) #modifico i null con la media
temp

In [ ]:
import os
import pandas as pd

folder_path = "/Users/valeriometelli/Desktop/Python - Epicode/datasets/beginner_datasets"
files = os.listdir(folder_path) #mi creo una lista di tutti i file contenuti nella cartella "Beginner_databes"

files_csv = [] #mi creo una lista per i file.csv

#faccio un ciclo for che va ad esaminare file per file se sono dei .csv, se lo sono li aggiunge in files_csv
for file in files: 
    if file.endswith(".csv"): 
        files_csv.append(file)

#creo una lista di dataframe, poi con un ciclo for esamino file per file nella lista files_csv
df_files = []
for file in files_csv:
    file_path = os.path.join(folder_path, file) #faccio join del path di beginner_datasets con il nome del file.csv
    df = pd.read_csv(file_path) #li legge uno per uno 
    df_files.append(df) #li aggiunge nella lista precedentemente creata

In [ ]:
#Dataset che contengono nulli
file_null = []
for i in range(len(df_files)):
    df = df_files[i]
    file_name = files_csv[i]
    if df.isnull().values.any():
        file_null.append(file_name)

print("File con valori nulli:")
print(file_null)

In [10]:
import os
import pandas as pd

#percorso della cartella
folder_path = "/Users/valeriometelli/Desktop/Python - Epicode/datasets/beginner_datasets"

#lista di tutti i file presenti nella cartella
files = os.listdir(folder_path)

#con un ciclo for popolo una lista con tutti i file.csv
files_csv = []
for file in files:
    if file.endswith(".csv"):
        files_csv.append(file)


df_csv = []    #creo lista nuova dove mettere i df
for file in files_csv:    #ciclo for su tutti i file.csv trovati
    file_path = os.path.join(folder_path, file)    #creo il percorso completo del file unendo cartella + nome file
    df = pd.read_csv(file_path)    #leggo il file CSV in un DataFrame pandas
    df_csv.append(df)    #aggiungo il df alla lista

for i in range(len(df_csv)): #ciclo for per controllare ogni df e il suo nome
    df = df_csv[i] #df corrente
    file_name = files_csv[i] #nome file corrente
    null_count = df.isna().sum().sum() #variabile che conta i dati null in ogni df
    if null_count > 0: #se i valori null sono > 0 allora...
        print(file_name, "ha", null_count, "valori nulli.")

nba.csv ha 11 valori nulli.
income.csv ha 4262 valori nulli.
hepatitis.csv ha 153 valori nulli.
seeds.csv ha 4 valori nulli.
france.csv ha 66 valori nulli.
traffic.csv ha 48143 valori nulli.
population.csv ha 12 valori nulli.
automobile.csv ha 39 valori nulli.
pokemon.csv ha 386 valori nulli.
wikipedia.csv ha 68 valori nulli.
house.csv ha 7829 valori nulli.
mice.csv ha 1396 valori nulli.
elections.csv ha 52 valori nulli.
